# LLM Steganography — Benchmarki (Colab, **online**)

Pierwsze uruchomienie / nowy model: wymaga **Hugging Face** (`HF_TOKEN`) i internetu.
Modele zapisują się na Drive w `models_cache/`.

> Tryb bez HF: użyj `Colab_Runner_Offline.ipynb` (tylko cache lokalny).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/Steganography_colab'
MODEL_CACHE_DIR = f'{PROJECT_DIR}/models_cache'

os.chdir(PROJECT_DIR)
os.environ['MODEL_CACHE_DIR'] = MODEL_CACHE_DIR
print('cwd:', Path.cwd())
print('model cache:', MODEL_CACHE_DIR)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
login(token=hf_token, add_to_git_credential=False)
print('HF login OK')

In [ ]:
required = ['run_experiments.py', 'dummy_processor.py']
missing = [name for name in required if not Path(name).exists()]
if missing:
    raise FileNotFoundError(f'Brakuje plikow w {PROJECT_DIR}: {missing}')
print('Wymagane pliki OK')

## Konfiguracja przebiegu

In [ ]:
# humaneval | perplexity | capacity | binoculars
TEST = 'humaneval'

# qwen | llama | gemma  (albo pelne HF model id)
MODEL = 'gemma'

# np. 0.01 albo 'all'
THRESHOLD = 0.0

TOP_N = 15
MAX_NEW_TOKENS = 256

# HumanEval only: None = wszystkie 164 zadania
# Pojedyncze: '5'  |  Zakres: '0-10'  |  Lista: '0,3,7'
HUMANEVAL_TASKS = None

# True = nie zapisuj modelu na dysk (tylko temp)
NO_MODEL_CACHE = False

In [ ]:
_humaneval_args = f' --humaneval-tasks {HUMANEVAL_TASKS}' if HUMANEVAL_TASKS else ''
_no_cache_args = ' --no-model-cache' if NO_MODEL_CACHE else ''

!python run_experiments.py \
  --test {TEST} \
  --model {MODEL} \
  --threshold {THRESHOLD} \
  --top-n {TOP_N} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --model-cache-dir {MODEL_CACHE_DIR}{_humaneval_args}{_no_cache_args}

In [ ]:
import csv

summary_csv = Path('results/summary.csv')
if summary_csv.exists():
    with summary_csv.open(newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print(f'Rows in summary.csv: {len(rows)}')
    for row in rows[-5:]:
        print(row)
else:
    print('summary.csv not found yet')